In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import sys
from pathlib import Path
sys.path.append(str(Path().absolute().parent))
from hcm_nn import hcm_nn, ASPP

input_tensor = torch.randn(1,3,256,256)
large_d = 256
large_input_tensor = torch.randn(1,3,large_d,large_d)
halved_tensor = torch.randn(1,3,112,112)
resnet50 = models.resnet50(pretrained=True)

def modify_resnet50_stride(model,stride_layer=2):
    # Modify the first convolutional layer
    model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=7, stride=1, padding=3, bias=False)
    
    # Modify the downsampling layers in the residual blocks
    strides = [1, 1, 1, 1]
    strides[stride_layer-1] = 1
    
    for i, layer in enumerate([model.layer1, model.layer2, model.layer3, model.layer4]):
        for block in layer:
            block.conv1.stride = (1, 1)
            block.conv2.stride = (1, 1)
            if block.downsample is not None:
                block.downsample[0].stride = (1, 1)
        if i < 4:
          layer[0].conv1.stride = (strides[i], strides[i])
          if layer[0].downsample is not None:
            layer[0].downsample[0].stride = (strides[i], strides[i])
    return model



# Modify the strides
#modified_model = modify_resnet50_stride(resnet50,2)

#resnet_features = nn.Sequential(*list(modified_model.children())[:-2])
#output_tensor = resnet_features(input_tensor)
#print("Resnet50 output shape:", output_tensor.shape)

#upsampled_output = F.interpolate(output_tensor,scale_factor=32, mode='bilinear')
#print("upsampled output shape:", upsampled_output.shape)

#repeated_tensor = halved_tensor.repeat(1,1,2,2)
#rt_mul = torch.mul(repeated_tensor,input_tensor)
#flipped_tensor = torch.flip(repeated_tensor,[2,3])
#ft_mul = torch.mul(flipped_tensor,input_tensor)
#cat_tensor = torch.cat((repeated_tensor,flipped_tensor),dim=3)

#conv = nn.Conv2d(3,3,kernel_size=1,padding=0,stride=(1,2))
#cat_tensor=conv(cat_tensor)

#print("repeated tensor size:",rt_mul.shape)
#print("flipped tensor shape:",ft_mul.shape)
#print("concatenated tensor shape:",cat_tensor.shape)
ASPP_RATES = [6,12,18]
hcm_model = hcm_nn(enable_finetuning=False)

output = hcm_model(input_tensor)

print("output from hcm shape:",output.shape)

print(f"Total model size: {hcm_model.get_model_size()}")


#model_size_mb = get_model_size(hcm_model)
#print(f"Model size: {model_size_mb:.2f} MB")

#aspp_model = ASPP(3,3,ASPP_RATES)

#output_aspp = aspp_model(large_input_tensor)

#print("Aspp model output shape:",output_aspp.shape)